# Control Variables - 02 Elevation
30/05/2026, Kuba Kowalski 

Bounding box data req: Xmin = -19.4765657186508	  Ymin = -37.76029790849866	  Xmax = 54.773438572883634	  Ymax = 36.55553077578038

Bounding box data req:-19.5, -37.8, 54.8, 36.6


In [11]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from pathlib import Path
import tempfile

# ------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------

province_polygons = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"
)

tiles_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\2_elevation\raw_tiles_srtmgl1"
)

out_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation"
)
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------

zone_id = "GEOLEVEL1"

control_var = "2a_elevation-sd"


In [ ]:
# Fix for incorrectly named tiles (messed up file extension due to missing period)

from pathlib import Path

tiles_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\ancilliary_data\2_elevation\raw_tiles_srtmgl1"
)

for file in tiles_dir.iterdir():
    if not file.is_file():
        continue

    old_name = file.name

    if old_name.endswith("ptif"):
        new_name = old_name[:-4] + ".tif"

    elif old_name.endswith("p.tif"):
        new_name = old_name[:-5] + ".tif"

    else:
        continue

    new_path = file.with_name(new_name)

    if new_path.exists():
        print(f"Skipping, already exists: {new_name}")
        continue

    file.rename(new_path)
    print(f"{old_name} -> {new_name}")

print("Done.")

print("Valid .tif files now:")
print(len(list(tiles_dir.glob("*.tif"))))

In [19]:
# ------------------------------------------------------------------
# LOAD PROVINCES
# ------------------------------------------------------------------

provinces = gpd.read_file(province_polygons)

if provinces.crs is None:
    raise ValueError("Province file has no CRS.")

provinces = provinces.to_crs("EPSG:4326")

if zone_id not in provinces.columns:
    raise ValueError(f"Column '{zone_id}' not found in province file.")

# ------------------------------------------------------------------
# LOAD DEM TILES
# ------------------------------------------------------------------

tile_files = sorted(tiles_dir.glob("*.tif"))

if len(tile_files) == 0:
    raise ValueError(f"No .tif files found in: {tiles_dir}")

print(f"Number of DEM tiles found: {len(tile_files)}")

src_files = [rasterio.open(fp) for fp in tile_files]

Number of DEM tiles found: 50


In [20]:
# ------------------------------------------------------------------
# MOSAIC DEM
# ------------------------------------------------------------------

print("Mosaicking DEM tiles...")

mosaic_array, mosaic_transform = merge(src_files)

mosaic_meta = src_files[0].meta.copy()
mosaic_meta.update({
    "driver": "GTiff",
    "height": mosaic_array.shape[1],
    "width": mosaic_array.shape[2],
    "transform": mosaic_transform,
    "count": 1,
    "dtype": mosaic_array.dtype,
    "crs": src_files[0].crs,
    "nodata": src_files[0].nodata
})

for src in src_files:
    src.close()

mosaic_file = out_dir / "srtm_africa_mosaic.tif"

with rasterio.open(mosaic_file, "w", **mosaic_meta) as dst:
    dst.write(mosaic_array)

print(f"Mosaic saved to: {mosaic_file}")

Mosaicking DEM tiles...
Mosaic saved to: C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\control_variables\2_elevation\srtm_africa_mosaic.tif


In [ ]:
# ------------------------------------------------------------------
# CALCULATE ELEVATION SD PER PROVINCE
# ------------------------------------------------------------------

results = []

with rasterio.open(mosaic_file) as src:

    nodata = src.nodata

    print("DEM CRS:", src.crs)
    print("DEM nodata:", nodata)

    if provinces.crs != src.crs:
        provinces = provinces.to_crs(src.crs)

    for idx, row in provinces.iterrows():

        province_id = row[zone_id]
        geom = [row.geometry]

        try:
            out_image, out_transform = mask(
                src,
                geom,
                crop=True,
                nodata=nodata,
                filled=True
            )

            values = out_image[0].astype("float64")

            if nodata is not None:
                values = values[values != nodata]

            values = values[np.isfinite(values)]

            # Optional: remove implausible SRTM fill values
            values = values[(values > -500) & (values < 9000)]

            if len(values) == 0:
                elev_sd = np.nan
                n_pixels = 0
            else:
                elev_sd = np.std(values, ddof=0)
                n_pixels = len(values)

        except Exception as e:
            print(f"Failed for {province_id}: {e}")
            elev_sd = np.nan
            n_pixels = 0

        results.append({
            zone_id: province_id,
            control_var: elev_sd,
            "2b_elevation-valid-pixels": n_pixels
        })

In [ ]:
# ------------------------------------------------------------------
# EXPORT TABLE
# ------------------------------------------------------------------

elevation_df = pd.DataFrame(results)

csv_path = out_dir / "2_elevation_sd.csv"
elevation_df.to_csv(csv_path, index=False)

print(f"CSV saved to: {csv_path}")

# ------------------------------------------------------------------
# JOIN BACK TO PROVINCES
# ------------------------------------------------------------------

provinces_elevation = provinces.merge(
    elevation_df,
    on=zone_id,
    how="left"
)

gpkg_path = out_dir / "2_elevation_sd.gpkg"
provinces_elevation.to_file(gpkg_path, driver="GPKG")

print(f"GPKG saved to: {gpkg_path}")

# ------------------------------------------------------------------
# SANITY CHECKS
# ------------------------------------------------------------------

print("\nMissing values:")
print(elevation_df.isna().sum())

print("\nSummary statistics:")
print(elevation_df[control_var].describe())

print("\nLowest elevation SD:")
print(elevation_df.sort_values(control_var).head(10))

print("\nHighest elevation SD:")
print(elevation_df.sort_values(control_var, ascending=False).head(10))

print("Done.")